# 6장 2강: 허깅페이스 모델 실습
## 2. 트랜스포머 아키텍처별 모델 실습


### 2.2 인코더 모델 활용: KoBERT로 감정 분류

In [1]:
# 토크나이저 및 모델 로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# KoBERT 토크나이저와 모델 로드
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("rkdaldus/ko-sent5-classification")

# 사용자 입력 텍스트 감정 분석
# text = "오늘 정말 행복해!"
text = "너무 무서워!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
print("inputs:", inputs)
with torch.no_grad():
    outputs = model(**inputs)
predicted_label = torch.argmax(outputs.logits, dim=1).item()

# 감정 레이블 정의
emotion_labels = {
    0: ("Angry", "😡"),
    1: ("Fear", "😨"),
    2: ("Happy", "😊"),
    3: ("Tender", "🥰"),
    4: ("Sad", "😢")
}

# 예측된 감정 출력
print(f"예측된 감정: {emotion_labels[predicted_label][0]} {emotion_labels[predicted_label][1]}")

c:\Users\bsis0\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6806.27it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: rkdaldus/ko-sent5-classification
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


inputs: {'input_ids': tensor([[   2, 1458, 2095, 6553, 7018,    5,    3]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
예측된 감정: Happy 😊


### 2.3 인코더-디코더 모델 활용: KoBART로 뉴스 요약

In [2]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

#text = "과거를 떠올려보자. 방송을 보던 우리의 모습을. 독보적인 매체는 TV였다. 온 가족이 둘러앉아 TV를 봤다. 간혹 가족들끼리 뉴스와 드라마, 예능 프로그램을 둘러싸고 리모컨 쟁탈전이 벌어지기도  했다. 각자 선호하는 프로그램을 ‘본방’으로 보기 위한 싸움이었다. TV가 한 대인지 두 대인지 여부도 그래서 중요했다. 지금은 어떤가. ‘안방극장’이라는 말은 옛말이 됐다. TV가 없는 집도 많다. 미디어의 혜 택을 누릴 수 있는 방법은 늘어났다. 각자의 방에서 각자의 휴대폰으로, 노트북으로, 태블릿으로 콘텐츠 를 즐긴다."
text = """신임 CEO 존 터너스는 이 과제를 풀 적임자로 꼽힌다. 2001년 입사 후 아이패드·에어팟·비전프로 개발을 주도한 하드웨어 엔지니어 출신이다. 업계에서는 애플이 소프트웨어만으로 AI 승부를 거는 대신, 자사의 강력한 강점인 ‘하드웨어+자체 칩+OS’ 결합 생태계에 AI를 이식하는 전략을 강화할 것으로 보고 있다.
터너스 체제의 첫 시험대는 당장 9일에 열리는 신제품 공개 행사다. 애플의 첫 폴더블 아이폰 등이 공개될 예정이다. 월스트리트저널(WSJ)은 “애플이 AI에서 혁신을 재점화해야 하는 과제를 안고 있다”고 전했다. 파이낸셜타임스는 “터너스의 성공은 애플의 하드웨어 강점과 AI 혁신을 얼마나 완벽하게 결합해 내느냐에 달렸다”고 평가했다.
"""

raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

summary_ids = model.generate(torch.tensor([input_ids]))
tokenizer.decode(summary_ids.squeeze().tolist(), skip_special_tokens=True)


Loading weights: 100%|██████████| 260/260 [00:00<00:00, 6558.41it/s]
c:\Users\bsis0\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\transformers\generation\utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'애플 CEO소프트웨어 엔지니어 출신 존 터너스가 애플의 하드웨어 강점과 AI 혁신을 얼마나 완벽하게'

### 2.4 디코더 모델 활용: Gemma로 대화형 텍스트 생성

In [ ]:
from transformers import pipeline
import torch

gemma_identifier = "google/gemma-2b-it"

gemma_generator = pipeline(
    "text-generation",
    model=gemma_identifier,
    dtype=torch.bfloat16, #양자화...?
    device_map="auto" # accelerate
#     top_p=
#     top_k=
#     temperature=
)

"""
role: system : 역할, 컨첵스트 ...
        user : 전달할 대화
        assistant: AI가 답변한 내용, 지난 대화 내용 + (user + assistant)

"""
user_dialogue = [
    {"role": "user", "content":"내 이름은 무엇이지?"}
]

outputs = gemma_generator(user_dialogue, max_new_tokens=150)
outputs

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\bsis0\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bsis0\.cache\huggingface\hub\models--google--gemma-2b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files:  50%|█████     | 

In [ ]:
user_dialogue = [
    {"role": "user", "content":"내 이름은 김철수야."},
    {"role": "assistant", "content":"반갑습니다. 철수님!"},
    {"role": "user", "content":"내 이름이 무엇이지?"}
]

outputs = gemma_generator(user_dialogue, max_new_tokens=150)
outputs